# Realtime EEG Labeling

This notebook labels generated realtime windows using `artifacts/realtime/` by default, or `EEG_OUTPUT_ROOT` when it is set. In Colab, either clone the repo into `/content` or set `EEG_PROJECT_ROOT` before running the first cell.

In [ ]:
from pathlib import Path
import os
import sys


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def is_repo_root(candidate: Path) -> bool:
    return (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'eeg_project').exists()


def dedupe_paths(paths):
    deduped = []
    seen = set()
    for path in paths:
        key = str(path)
        if key not in seen:
            deduped.append(path)
            seen.add(key)
    return deduped


def candidate_repo_roots():
    roots = []

    env_root = os.environ.get('EEG_PROJECT_ROOT')
    if env_root:
        roots.append(Path(env_root).expanduser())

    roots.extend([Path.cwd(), *Path.cwd().parents])

    if running_in_colab():
        roots.extend([
            Path('/content'),
            Path('/content/eeg-Spr2026-CSCI7090'),
            Path('/content/drive/MyDrive'),
            Path('/content/drive/MyDrive/eeg-Spr2026-CSCI7090'),
            Path('/content/drive/MyDrive/Colab Notebooks'),
            Path('/content/drive/Shareddrives'),
        ])

    roots.extend([
        Path.home(),
        Path.home() / 'Developer',
        Path.home() / 'Documents',
    ])
    return dedupe_paths(path.expanduser() for path in roots)


def discover_repo_root() -> Path:
    checked = []

    for root in candidate_repo_roots():
        for candidate in [root, *root.parents]:
            checked.append(candidate)
            if is_repo_root(candidate):
                return candidate.resolve()

    search_roots = [root for root in candidate_repo_roots() if root.exists()]
    matches_checked = 0
    for search_root in search_roots:
        for pyproject_file in search_root.rglob('pyproject.toml'):
            matches_checked += 1
            candidate = pyproject_file.parent
            checked.append(candidate)
            if is_repo_root(candidate):
                return candidate.resolve()
            if matches_checked >= 300:
                break
        if matches_checked >= 300:
            break

    checked_display = '\n'.join(f' - {path}' for path in dedupe_paths(checked)[:20])
    raise ModuleNotFoundError(
        'Could not locate the repository root.\n'
        'If you are in Colab, either clone the repo into /content or set EEG_PROJECT_ROOT.\n'
        'Example:\n'
        "os.environ['EEG_PROJECT_ROOT'] = '/content/eeg-Spr2026-CSCI7090'\n\n"
        f'Checked:\n{checked_display}'
    )


def ensure_repo_src_on_path() -> Path:
    repo_root = discover_repo_root()
    src_dir = (repo_root / 'src').resolve()
    resolved = str(src_dir)
    if resolved not in sys.path:
        sys.path.insert(0, resolved)
    return src_dir


src_dir = ensure_repo_src_on_path()
print('Using source path:', src_dir)

from eeg_project.realtime import get_realtime_paths, load_feature_matrix, run_phase2_labeling

paths = get_realtime_paths()
print('Manifest path:', paths.manifest_path)
print('Annotations path:', paths.annotations_path)

In [ ]:
manifest_df, annotations_df, labeled_df, training_df = run_phase2_labeling()
print('Raw rows:', len(manifest_df))
print('Labeled rows:', len(labeled_df))
print('Training rows:', len(training_df))
training_df.head()

In [ ]:
import matplotlib.pyplot as plt

labeled_df['label'].value_counts().plot(kind='bar')
plt.title('Window Counts by Label')
plt.xlabel('Label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
X, y = load_feature_matrix(training_df)
print('Feature matrix shape:', X.shape)
print('Target distribution:')
print(training_df['target'].value_counts())